In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import cv2
import time
import io
from IPython import display
from PIL import Image
from tqdm import tqdm
from simulator.run_analysis import *

### ---- Analyse all Trajectories and delete bad ones -------

In [ ]:
show_all_plots('/home/alex/flex/MCMD_Sim/results_accepted', plot_tag='sim_pos_xy')

In [ ]:
# bad_runs = [f for f in os.listdir('/home/alex/flex/MCMD_Sim/results') if f.startswith('1')]
# len(bad_runs)

In [ ]:
# delete_datas('/home/alex/flex/BLIP2_DATASET', '/home/alex/flex/MCMD_Sim/results', bad_runs)

In [ ]:
watch_videos('/home/alex/flex/MCMD_Sim/results_accepted', speed=1000.0, wait_time=0.1, resize=(400, 400))

In [ ]:
from datetime import datetime

### --------Viewing and Changing instruction Text------------------

In [ ]:
def get_all_texts(data_base, run_types=None):
    all_texts = {'train': {}, 'eval': {}}
    if run_types is None: run_types = os.listdir(os.path.join(data_base, 'train'))
    for run_for in ['train', 'eval']:
        for run_type in run_types:
            for run in os.listdir(os.path.join(data_base, run_for, run_type)):
                with open(os.path.join(data_base, run_for, run_type, run, 'label.txt'), 'r') as f:
                    text = f.read()
                all_texts[run_for][run] = text
    return all_texts


def change_run_text(data_base, gen_inst_func, dcommand = None, dcolor = None):
    runs = os.listdir(data_base)
    log_path =  f'label_log_{datetime.now().strftime("%Y_%m_%d_%H_%M_%S")}.txt'
    with open(log_path, 'a') as log_file:
        log_file.write(f"\n# Change session at {datetime.now()}\n")
        for run in runs:
            run_path = os.path.join(data_base, run)
            with open(os.path.join(run_path, 'label.txt'), 'r') as f:
                old_text = f.read() 
            if run.startswith('save'):
                command = 'towards'
                text = old_text.split(' ')
                color = 'red' if 'red' in text else 'blue'
                if dcolor is not None: color = dcolor
            else:
                run_type = run.split('_')
                command, color = dcommand or run_type[0], dcolor or run_type[1][1:]
            new_text = gen_inst_func(color=color, direction=command, obj_type = 'ball')
            with open(os.path.join(run_path, 'label.txt'), 'w') as f:
                f.write(new_text)
            log_file.write(f'{run_path} | FROM: "{old_text}" | TO: "{new_text}"\n')
            print(f"Changed {run} from {old_text} to {new_text}")

In [ ]:
def generate_text(color='blue', direction='towards', obj_type = 'ball'):
    obj = f'{color} {obj_type}' if color not in {'', None} else obj_type
    return f'{direction}---{obj}'

In [ ]:
change_run_text('/home/alex/flex/BLIP2_DATASET/train/above_red_new', generate_text)

In [ ]:
get_all_texts('/home/alex/flex/BLIP2_DATASET/')

In [ ]:
os.listdir('/home/alex/flex/BLIP2_DATASET/train')

### ---------------Convert to features -----------------------

In [ ]:
import os
import torch
import random
import numpy as np
from tqdm import tqdm
from PIL import Image
import sys
sys.path.append('/home/alex/flex/learning/src/models/components/extractors')
from blip import BLIPExtractor # type: ignore
from torchvision import transforms # type: ignore


# ----------------- SEED FUNCTION -----------------
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)

# ----------------- DATA to BLIP FUNCTION -----------------
def data2BLIP(model, data_path, target_path, device, run_types = None):
    if run_types is None: run_types = sorted(os.listdir(data_path))
    for run_type in run_types:
        print(f"Processing {run_type}")
        runs = sorted(os.listdir(os.path.join(data_path, run_type)))
        for i, run in tqdm(enumerate(runs), leave=True, total=len(runs)):
            target_path_run = os.path.join(target_path, run_type, run)
            os.makedirs(target_path_run, exist_ok=True)

            run = os.path.join(data_path, run_type, run)
            with open(os.path.join(run,  'label.txt'), 'r') as f:
                text = f.read()
            text = text.split('---')[1]

            image_names = sorted([f for f in os.listdir(run) if f.endswith('.png')])
            run_loop = tqdm(enumerate(image_names), total=len(image_names), desc=f"Run {i+1}/{len(runs)}", leave=False)
            for j, im_name in run_loop:
                img = Image.open(os.path.join(run, im_name))
                img = img.convert('RGB')
                img = img.resize((224, 224))
                img = transforms.ToTensor()(img)
                data_out = model({"image": img.to(device), "text":text})
                torch.save(data_out.cpu(), os.path.join(target_path_run, im_name.replace('.png', '.pt')))
                run_loop.set_postfix({'Image': im_name})

def run_extraction(run_types = None):
    data_path:str = '/home/alex/flex/BLIP2_DATASET/'
    target_dir:str = "/home/alex/flex/BLIP2_Features/"
    device = torch.device('cuda:0')
    set_seed(42)

    model = BLIPExtractor(
            'blip2_feature_extractor',
            'pretrain',
            freeze_blip=True,
            use_low_dim_feature=False,
            last_linear_layer=None,      #features before last linear layer
            use_continuous_pe=False,
            stride=14,
            checkpoint='~/.cache/torch/hub/checkpoints/blip2_pretrained.pth',
            patch_size=2,
            all_q_dims= False,
            use_masked_patch_wise_feature= True,
            use_visual_encoder_only= False
        )

    model.to(device)
    model.eval()
    data2BLIP(model, data_path+'train/', target_dir+'train/', device, run_types)
    data2BLIP(model, data_path+'eval/', target_dir+'eval/', device, run_types)

In [ ]:
run_extraction(['below_red_new', 'above_red_new', 'below_blue_new', 'above_blue_new'])

### ------- Trajectory Math -----------

In [ ]:
from simulator.utils import  SimUtils
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from simulator.utils.logger import SimLogger
from simulator.components.simobjects import SimObject

In [ ]:
def calc_ang(ref, P1, P2):
    v1 = P1[:2] - ref[:2]
    v2 = P2[:2] - ref[:2]
    ang = np.arccos(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))
    return ang

def ang_xy(v1, v2):
    v1, v2 = v1[:2], v2[:2]
    ang = np.arccos(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))
    return ang

In [ ]:
def plot_trajectory(pos_data, yaw_data, theta_env, objs=None):        
    pos_id = {'x': 0, 'y':1, 'z':2}
    for i, sbplt in enumerate(['xy', 'yz', 'xz']):
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.plot(pos_data[:, pos_id[sbplt[0]]], pos_data[:, pos_id[sbplt[1]]], marker='o', alpha=0.5, color='blue')
        if sbplt == 'xy':
            for i in range(0, yaw_data.shape[0], 5):
                x, y = pos_data[i, pos_id['x']], pos_data[i, pos_id['y']]
                ang = yaw_data[i] - theta_env + 2* np.pi
                dx, dy = 0.13 * np.cos(ang), 0.13 * np.sin(ang)
                ax.arrow(x, y, dx, dy, head_width=0.02, head_length=0.02, color='red')
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel(sbplt[0])
        ax.set_ylabel(sbplt[1])
        ax.set_title(f'{sbplt} Graph with Objects')
        if objs is not None:
            for obj in objs:
                ax.plot(obj[sbplt[0]], obj[sbplt[1]], obj['style'])
                circle = Circle((obj[sbplt[0]], obj[sbplt[1]]), 0.5, color='red', fill=False, linestyle='--')
                ax.add_patch(circle)
        fig.show()
        # plt.close(fig)

    fig, ax = plt.subplots()
    ax.plot(yaw_data)
    ax.set_ylabel('Yaw')
    fig.show()

In [ ]:
class SimAnalyzer:
    def __init__(self, traj_path):
        self.traj_path = traj_path
        init_cond = json.loads(open(os.path.join(traj_path, 'init_conditions.json'), 'r').read())
        self.theta_offset = init_cond["theta_offset"]
        self.theta_env = init_cond["theta_environment"]
        self.objs = [
            SimObject(loc_xy, self.theta_env, colr, obj_type) 
            for loc_xy, colr, obj_type in zip(
                init_cond['objects_loc'], 
                init_cond["objects_color"], 
                init_cond['objects_type']
            )
        ]
        self.drone = SimObject(
            init_cond['drones_loc'][0], 
            self.theta_env, 
            obj_type='drone', colr='black'
        )
        self.target = init_cond['target_idx']
        with open(os.path.join(traj_path, 'logs.txt'), 'r') as f:
            text = f.read()
        text = text.split('\n')[1].replace('(', '').split(' ')[-3:]
        self.Pd_rel = np.array([float(x[:-1]) for x in text])

In [ ]:
traj_path = '/home/alex/flex/MCMD_Sim/results/left_1red_2025_03_29_21_58_48'

In [ ]:
sim = SimAnalyzer(traj_path)

In [ ]:
# target = 1 
# Pd_rel = np.array([Ptr[0], 0, Ptr[2]])

In [ ]:
target = sim.target
Ptr = sim.objs[target].loc_rel
Pd_rel = sim.Pd_rel

In [ ]:
Pt = np.array(sim.objs[target].loc_abs)
P0 = np.array(sim.drone.loc_abs)
Pd = np.array(SimUtils.convert_to_global(Pd_rel, sim.objs[1].theta))

In [ ]:
n_points = 8800
c_steps = np.linspace(0, 1, n_points)
c_steps = np.column_stack([
    1 - (1 - c_steps)**3,
    1 - (1 - c_steps)**5,
    1 - (1 - c_steps)**20
])
t_yaw = np.linspace(0, 1, n_points) ** 1.6

c_dir = Pd - Pt
d_dir = Pd - P0

c_values = 0.5 * np.sin(np.pi * c_steps) 
trajectory = P0[None, :] + c_steps * d_dir[None, :] + c_values * c_dir

c_ortho = np.array([c_dir[1], -c_dir[0], 0])
sgn = np.sign(np.dot(c_ortho, d_dir))
dd = 0.8 * c_dir + 2 * c_ortho * sgn 

delta_xy = Pt[None, :2] + t_yaw[:, None] * dd[None, :2] - trajectory[:, :2]
yaw_angles = np.unwrap(np.arctan2(delta_xy[:, 1], delta_xy[:, 0]))

In [ ]:
Po = dd + Pt
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(P0[0], P0[1], color='green', label='P0', s=100)
ax.scatter(Pt[0], Pt[1], color='blue', label='Pt', s=100)
ax.scatter(Pd[0], Pd[1], color='orange', label='Pd', s=100)
ax.scatter(Po[0], Po[1], color='red', label='Po', s=100)

ax.plot([P0[0], Pt[0], Pd[0], Po[0]], [P0[1], Pt[1], Pd[1], Po[1]], linestyle='--', color='gray', label='P0 - Pt - Pd - Po')
ax.plot([P0[0], Pd[0]], [P0[1], Pd[1]], linestyle='--', color='purple', label='P0 - Pd')
ax.plot([P0[0], Po[0]], [P0[1], Po[1]], linestyle='--', color='brown', label='P0 - Po')
# ax.plot([Po[0], Pt[0]], [Po[1], Pt[1]], linestyle='--', color='pink', label='Po to Pt')

ax.legend()

plt.show()
ang_t0d = calc_ang(P0[:2], Pt[:2], Pd[:2])
ang_t0o = calc_ang(P0[:2], Pt[:2], Po)
np.sign(ang_t0d -ang_t0o)

In [ ]:
plot_trajectory(
    # SimUtils.traj2xyz_relative_to_base_env(trajectory[::50], sim.theta_env), 
    trajectory[::50],
    yaw_angles[::50], 
    0,
    # sim.theta_env, 
    [SimLogger.parse_obj(obj, False) for obj in sim.objs]
)

In [ ]:
dataset_base = '/home/alex/flex/BLIP2_DATASET/train'
plots_base = '/home/alex/flex/MCMD_Sim/results'
run_folders = os.listdir(dataset_base)
summary_data = []

In [ ]:
all_train_sf_labels = []
for run in os.listdir(os.path.join(dataset_base, 'save_flight')):
    with open(os.path.join(dataset_base, 'save_flight', run, 'label.txt'), 'r') as f:
        all_train_sf_labels.append(f.read())
with open('all_train_sf_labels.txt', 'w') as f:
    f.write('\n'.join(all_train_sf_labels))


In [ ]:
all_train_sf_labels_lst = []
for sf_label in all_train_sf_labels:
    sf_label_split = sf_label.split(' ')
    for i, sf_label_wrd in enumerate(sf_label_split):
        if i >= len(all_train_sf_labels_lst): all_train_sf_labels_lst.append(set())
        all_train_sf_labels_lst[i].add(sf_label_wrd)

In [ ]:
all_train_sf_labels_lst[5]

In [ ]:
# for run in run_folders:
#     data_path = os.path.join(dataset_base, run, 'data_out.csv')
#     df = pd.read_csv(data_path)
#     vx, vy, vz, yr = df.iloc[:, :4].values.T
#     row_data = {
#         'run': run,
#         'vx_mean': vx.mean(),
#         'vy_mean': vy.mean(),
#         'vz_mean': vz.mean(),
#         'yr_mean': yr.mean(),
#         'vx_std': vx.std(),
#         'vy_std': vy.std(),
#         'vz_std': vz.std(),
#         'yr_std': yr.std(),
#         'total_length': len(df)
#     }
#     summary_data.append(row_data)

# summary_df = pd.DataFrame(summary_data)
    

In [ ]:
# summary_df[['run', 'total_length']].sort_values('total_length', ascending=False).iloc[-15:]

In [ ]:
# summary_df[['total_length']].plot(kind='line', marker='o')
# plt.title("Mean Values per Run")
# plt.show()

In [ ]:
def show_all_plots(plots_base, plot_tag='sim_pos_xy'):
    plots_folders = sorted(os.listdir(plots_base))
    for i in range(0, len(plots_folders), 25):
        fig, axes = plt.subplots(5, 5, figsize=(20, 20))
        for j in range(5):
            for k in range(5):
                if i+j*5+k >= len(plots_folders):
                    break
                run = plots_folders[i+j*5+k]
                splots = [sp for sp in os.listdir(os.path.join(plots_base, run)) if sp.startswith(plot_tag)]
                im_path = os.path.join(plots_base, run, splots[0])
                axes[j, k].imshow(plt.imread(im_path))
                axes[j, k].set_title(run)
        plt.show()

In [ ]:
import cv2
import IPython.display as display
from PIL import Image
import io

cap = cv2.VideoCapture('/home/alex/flex/MCMD_Sim/results/2025_03_17_12_27_38/out_video_12_27_38.mp4')

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    _, buffer = cv2.imencode('.jpeg', frame)
    display.display(Image.open(io.BytesIO(buffer)))
    display.clear_output(wait=True)

cap.release()

In [ ]:
import cv2
import time

def play_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not open video.")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_number = 0
    speed = 1.0  # Default speed (1x)
    paused = False
    repeat = True  # Enable repeat mode

    while True:
        if not paused:
            ret, frame = cap.read()
            if not ret:
                if repeat:
                    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                    frame_number = 0
                    continue
                else:
                    break  # Exit when video ends

            # Display the frame number
            cv2.putText(frame, f'Frame: {frame_number}/{total_frames}', (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # cv2.imshow('Video Player', frame)
            _, buffer = cv2.imencode('.jpeg', frame)
            display.display(Image.open(io.BytesIO(buffer)))
            display.clear_output(wait=True)
            time.sleep(1 / speed)
            frame_number += 1

        # Keyboard Controls
        # key = cv2.waitKey(int(50 / speed)) & 0xFF
        # if key == ord('q'):  # Quit
        #     break
        # elif key == ord('p'):  # Pause/Resume
        #     paused = not paused
        # elif key == ord('r'):  # Restart Video
        #     cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        #     frame_number = 0
        # elif key == ord('+'):  # Increase Speed
        #     speed = min(3.0, speed + 0.1)
        # elif key == ord('-'):  # Decrease Speed
        #     speed = max(0.1, speed - 0.1)

    cap.release()
    # cv2.destroyAllWindows()


In [ ]:
video_path = '/home/alex/flex/MCMD_Sim/results/left_0red_2025_03_26_16_15_41/out_video_3_26_16_15_41.mp4'  # Replace with your video path
play_video(video_path)

In [ ]:
import torch
import numpy as np
import time

In [ ]:
tt = torch.randn(1, 768, 8, 8)

In [ ]:
# File paths for saving
torch_file = 'tensor_torch.pt'
torch_file_compressed = 'tensor_torch_compressed.pt'
numpy_file = 'tensor_numpy.npy'
numpy_file_compressed = 'tensor_numpy_compressed.npz'

# Save the tensor using PyTorch
torch.save(tt, torch_file)
torch.save(tt, torch_file_compressed, _use_new_zipfile_serialization=True)

# Save the tensor using NumPy
np.save(numpy_file, tt.numpy())
np.savez_compressed(numpy_file_compressed, data=tt.numpy())

# Loading function with timing
def load_with_time(load_func, path):
    start = time.time()
    data = load_func(path)
    end = time.time()
    return data, end - start

# Load and measure time
torch_data, torch_time = load_with_time(torch.load, torch_file)
torch_compressed_data, torch_compressed_time = load_with_time(torch.load, torch_file_compressed)

numpy_data, numpy_time = load_with_time(lambda p: torch.from_numpy(np.load(p)), numpy_file)
numpy_compressed_data, numpy_compressed_time = load_with_time(lambda p: torch.from_numpy(np.load(p)['data']), numpy_file_compressed)

# Comparison
def compare_tensors(t1, t2):
    return torch.allclose(torch.tensor(t1), torch.tensor(t2), atol=1e-6)

print(f"Torch (Uncompressed) Load Time: {torch_time:.4f} sec")
print(f"Torch (Compressed) Load Time: {torch_compressed_time:.4f} sec")
print(f"NumPy (Uncompressed) Load Time: {numpy_time:.4f} sec")
print(f"NumPy (Compressed) Load Time: {numpy_compressed_time:.4f} sec")

print("\nData Integrity Check:")
print(f"Torch (Uncompressed) == Original: {compare_tensors(tt, torch_data)}")
print(f"Torch (Compressed) == Original: {compare_tensors(tt, torch_compressed_data)}")
print(f"NumPy (Uncompressed) == Original: {compare_tensors(tt, numpy_data)}")
print(f"NumPy (Compressed) == Original: {compare_tensors(tt, numpy_compressed_data)}")


In [ ]:
start = time.time()
tt2 = torch.load('/home/alex/flex/BLIP2_Features/train/save-flight-02.13.2024_21.38.25.755537/000000a.pth', weights_only=False)
print(time.time() - start)

In [ ]:
tt2.device

In [ ]:
tt2.dtype

In [ ]:
torch.save(tt2, 'tt2.pt')

In [ ]:
torch.save(tt2.cpu(), 'tt2.pt')

In [ ]:
start = time.time()
tt3 = torch.load('tt2.pt', weights_only=False, map_location='cpu')
print(time.time() - start)

In [ ]:
(tt3.to(tt2.device) - tt2).sum()

In [ ]:
import os
from tqdm import tqdm

In [ ]:
base_dir = '/home/alex/flex/BLIP2_Features'
new_dir = '/home/alex/flex/BLIP2_Features2'


In [ ]:
l1_dir = os.listdir(base_dir)
for l1 in l1_dir:
    l2_dir = os.listdir(os.path.join(base_dir, l1))
    for l2 in tqdm(l2_dir, desc=l1):
        all_files = [f for f in os.listdir(os.path.join(base_dir, l1, l2)) if f.endswith('.pth')]
        os.makedirs(os.path.join(new_dir, l1, l2), exist_ok=True)
        for f in tqdm(all_files, desc=l2, leave=False):
            tt = torch.load(os.path.join(base_dir, l1, l2, f), weights_only=False)
            torch.save(tt.cpu(), os.path.join(new_dir, l1, l2, f.replace('.pth', '.pt')))